<a href="https://colab.research.google.com/github/Jasonmiller513/Dissertation/blob/Updates/Copy_of_RL_Process.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **STEP 1: Remove unnecessary columns and clean data**

# Import Extensions And Open File

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
import joblib
from collections import defaultdict
df = pd.read_csv('DataCoSupplyChainDataset.csv', encoding='latin1')

# Remove Unnecessary Columns

In [3]:
# List of columns to remove
columns_to_remove = [

    # Identifier columns
    'Customer Id',
    'Category Id',
    'Department Id',

    # Personal/customer detail columns
    'Customer Email',
    'Customer Password',
    'Customer Fname',
    'Customer Lname',
    'Customer Street',
    'Customer Zipcode',
    'Customer Full Name',

    # High-cardinality text columns
    'Product Description',
    'Product Image',
    'Product Name',


    # Duplicate or Redundant Geographic Columns

'Customer Segment',
'Order City',
'Order State',]

# Remove only columns that actually exist in dataframe
existing_columns_to_remove = [
    col for col in columns_to_remove if col in df.columns]

# Drop columns
df_rl = df.drop(columns=existing_columns_to_remove)

# Display remaining columns
print("Remaining Columns:")
print(df_rl.columns.tolist())

# Display shape after removal
print("\nNew Dataset Shape:")
print(df_rl.shape)

Remaining Columns:
['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Name', 'Customer City', 'Customer Country', 'Customer State', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order Country', 'Order Customer Id', 'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order Status', 'Order Zipcode', 'Product Card Id', 'Product Category Id', 'Product Price', 'Product Status', 'shipping date (DateOrders)', 'Shipping Mode']

New Dataset Shape:
(180519, 38)


In [4]:
# Save cleaned RL dataset to CSV in Google Colab

output_file = 'dataco_rl_cleaned.csv'

# Save dataframe
df_rl.to_csv(output_file, index=False)

print(f"Dataset saved as: {output_file}")

Dataset saved as: dataco_rl_cleaned.csv


# Identify missing or erroneous data


In [5]:
df = pd.read_csv('dataco_rl_cleaned.csv')

print(df.shape)
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0])

(180519, 38)
Order Zipcode    155679
dtype: int64


In [6]:
duplicate_count = df.duplicated().sum()
print(f"Duplicate Rows: {duplicate_count}")

Duplicate Rows: 0


In [7]:
numeric_cols = df.select_dtypes(include=np.number).columns

# Display summary statistics
print(df[numeric_cols].describe())

       Days for shipping (real)  Days for shipment (scheduled)  \
count             180519.000000                  180519.000000   
mean                   3.497654                       2.931847   
std                    1.623722                       1.374449   
min                    0.000000                       0.000000   
25%                    2.000000                       2.000000   
50%                    3.000000                       4.000000   
75%                    5.000000                       4.000000   
max                    6.000000                       4.000000   

       Benefit per order  Sales per customer  Late_delivery_risk  \
count      180519.000000       180519.000000       180519.000000   
mean           21.974989          183.107609            0.548291   
std           104.433526          120.043670            0.497664   
min         -4274.979980            7.490000            0.000000   
25%             7.000000          104.379997            0.000000 

In [8]:
non_negative_columns = [
    'Sales',
    'Order Item Quantity',
    'Order Item Total',
    'Product Price',
    'Days for shipping (scheduled)']

for col in non_negative_columns:

    if col in df.columns:

        # Count invalid negatives
        invalid_count = (df[col] < 0).sum()

        print(f"\n{col} Negative Values: {invalid_count}")

        # Replace negatives with median
        median_value = df[df[col] >= 0][col].median()

        df.loc[df[col] < 0, col] = median_value


Sales Negative Values: 0

Order Item Quantity Negative Values: 0

Order Item Total Negative Values: 0

Product Price Negative Values: 0


In [9]:
for col in numeric_cols:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outlier_count = (
        ((df[col] < lower_bound) |
         (df[col] > upper_bound))
    ).sum()

    print(f"{col}: {outlier_count} outliers")

Days for shipping (real): 0 outliers
Days for shipment (scheduled): 0 outliers
Benefit per order: 18942 outliers
Sales per customer: 1943 outliers
Late_delivery_risk: 0 outliers
Latitude: 9 outliers
Longitude: 1414 outliers
Order Customer Id: 1198 outliers
Order Id: 0 outliers
Order Item Cardprod Id: 0 outliers
Order Item Discount: 7537 outliers
Order Item Discount Rate: 0 outliers
Order Item Id: 0 outliers
Order Item Product Price: 2048 outliers
Order Item Profit Ratio: 17300 outliers
Order Item Quantity: 0 outliers
Sales: 488 outliers
Order Item Total: 1943 outliers
Order Profit Per Order: 18942 outliers
Order Zipcode: 0 outliers
Product Card Id: 0 outliers
Product Category Id: 0 outliers
Product Price: 2048 outliers
Product Status: 0 outliers


In [10]:
print("\nRemaining Missing Values:")
print(df.isnull().sum().sum())

print("\nFinal Dataset Shape:")
print(df.shape)

print("\nDataset Preview:")
print(df.head())


Remaining Missing Values:
155679

Final Dataset Shape:
(180519, 38)

Dataset Preview:
       Type  Days for shipping (real)  Days for shipment (scheduled)  \
0     DEBIT                         3                              4   
1  TRANSFER                         5                              4   
2      CASH                         4                              4   
3     DEBIT                         3                              4   
4   PAYMENT                         2                              4   

   Benefit per order  Sales per customer   Delivery Status  \
0          91.250000          314.640015  Advance shipping   
1        -249.089996          311.359985     Late delivery   
2        -247.779999          309.720001  Shipping on time   
3          22.860001          304.809998  Advance shipping   
4         134.210007          298.250000  Advance shipping   

   Late_delivery_risk   Category Name Customer City Customer Country  ...  \
0                   0  Sportin

# **STEP 2: FEATURE ENGINEERING**

In [11]:
df = pd.read_csv('dataco_rl_cleaned.csv', encoding='latin1')

print("Dataset Loaded")
print(df.shape)

Dataset Loaded
(180519, 38)


# Date and Time Features

In [12]:
# Convert order date to datetime
df['order date (DateOrders)'] = pd.to_datetime(
    df['order date (DateOrders)'],
    errors='coerce')

# Extract temporal features
df['order_year'] = df['order date (DateOrders)'].dt.year
df['order_month'] = df['order date (DateOrders)'].dt.month
df['order_day'] = df['order date (DateOrders)'].dt.day
df['order_dayofweek'] = df['order date (DateOrders)'].dt.dayofweek
df['is_weekend'] = df['order_dayofweek'].isin([5,6]).astype(int)


# Create a Product-Region Matrix and Network

In [13]:
# Create a Product-Region Matrix
product_region = (
    df.groupby("Product Card Id")["Order Region"]
      .nunique()
      .reset_index(name="num_regions")
      .sort_values("num_regions", ascending=False))

print(product_region.head(20))

    Product Card Id  num_regions
10              116           23
21              249           23
14              172           23
15              191           23
9                93           23
25              276           23
38              564           23
40              567           23
44              627           23
35              365           23
36              403           23
37              502           23
53              703           23
45              642           23
24              273           23
87              905           23
83              885           23
85              893           23
76              823           23
75              822           23


In [14]:
order_features = (
    df.groupby("Order Id")
      .agg(
          unique_products=("Product Card Id", "nunique"),
          total_quantity=("Order Item Quantity", "sum"),
          total_sales=("Sales", "sum"))
      .reset_index())

order_features["complex_order"] = (
    order_features["unique_products"] >= 3
).astype(int)

In [15]:
network = (
    df.groupby(["Product Card Id", "Order Region"])
      .agg(
          orders=("Order Id", "nunique"),
          quantity=("Order Item Quantity", "sum"),
          revenue=("Sales", "sum"))
      .reset_index())

# Explore Product Statistics

In [16]:
product_stats = (
    network.groupby("Product Card Id")
           .agg(
               regions_served=("Order Region", "nunique"),
               total_orders=("orders", "sum"),
               total_quantity=("quantity", "sum"),
               total_revenue=("revenue", "sum"))
           .reset_index())


# Define Multi-Region Stock

In [17]:
region_threshold = product_stats["regions_served"].median()
order_threshold = product_stats["total_orders"].median()

product_stats["likely_multi_region_stocked"] = (
    (product_stats["regions_served"] >= region_threshold)
    &
    (product_stats["total_orders"] >= order_threshold)
).astype(int)

#Create a stock score

In [18]:
product_stats["stocking_score"] = (
    0.5 *
    (
        product_stats["regions_served"]
        / product_stats["regions_served"].max())
    +
    0.5 *
    (
        product_stats["total_orders"]
        / product_stats["total_orders"].max()))


# Merge the network

In [19]:
network = network.merge(
    product_stats[
        [
            "Product Card Id",
            "regions_served",
            "stocking_score",
            "likely_multi_region_stocked"
        ]
    ],
    on="Product Card Id",
    how="left")

# Find Edge Weight

In [20]:
network["edge_weight"] = (
    0.4 *
    (network["orders"] / network["orders"].max())
    +
    0.3 *
    (network["quantity"] / network["quantity"].max())
    +
    0.3 *
    (network["revenue"] / network["revenue"].max()))


 # Find candidate shipping regions

In [21]:
candidate_regions = (
    network.sort_values(
        ["Product Card Id", "edge_weight"],
        ascending=[True, False])
    .groupby("Product Card Id")
    .head(5))

# Save Outputs

In [22]:
network.to_csv(
    "product_region_network.csv",
    index=False
)

candidate_regions.to_csv(
    "candidate_fulfillment_regions.csv",
    index=False
)

product_stats.to_csv(
    "product_stocking_scores.csv",
    index=False
)

print("Files created successfully")

Files created successfully


# Shipping Engineering

In [24]:
df['order date (DateOrders)'] = pd.to_datetime(
    df['order date (DateOrders)'],
    errors='coerce'
)
df['shipping date (DateOrders)'] = pd.to_datetime(
    df['shipping date (DateOrders)'],
    errors='coerce'
)

# Actual shipping time

df['actual_ship_days'] = (
    df['shipping date (DateOrders)']
    - df['order date (DateOrders)']
).dt.days

# Difference from scheduled

df['shipping_delay'] = (
    df['Days for shipping (real)']
    - df['Days for shipment (scheduled)']
)

# Binary on-time flag

df['on_time_delivery'] = (
    df['shipping_delay'] <= 0
).astype(int)

# Late shipment flag

df['late_delivery'] = (
    df['shipping_delay'] > 0
).astype(int)

# Profit Engineering

In [25]:
# Margin percentage

df['margin_pct'] = (
    df['Order Profit Per Order']
    / df['Sales']
)

# Profit per item

df['profit_per_item'] = (
    df['Order Profit Per Order']
    / df['Order Item Quantity']
)

# Discount percentage

df['discount_pct'] = (
    df['Order Item Discount']
    / df['Order Item Product Price']
)

# Product Demand Features

In [26]:
product_stats = (
    df.groupby('Product Card Id')
      .agg(
          product_orders=('Order Id','nunique'),
          product_quantity=('Order Item Quantity','sum'),
          product_revenue=('Sales','sum')
      )
      .reset_index()
)

df = df.merge(
    product_stats,
    on='Product Card Id',
    how='left'
)

# Regional Features

In [27]:
region_stats = (
    df.groupby('Order Region')
      .agg(
          region_orders=('Order Id','nunique'),
          region_sales=('Sales','sum'),
          region_profit=('Order Profit Per Order','sum')
      )
      .reset_index()
)

df = df.merge(
    region_stats,
    on='Order Region',
    how='left'
)

# Fulfillment Network Features

In [28]:
stocking = pd.read_csv(
    "product_stocking_scores.csv"
)

df = df.merge(
    stocking[
        [
            'Product Card Id',
            'stocking_score',
            'regions_served',
            'likely_multi_region_stocked'
        ]
    ],
    on='Product Card Id',
    how='left'
)

# Shipping Mode Features

In [29]:
df['shipping_mode_speed'] = df['Shipping Mode'].map({
    'Same Day':4,
    'First Class':3,
    'Second Class':2,
    'Standard Class':1
})

# Order Complexity Features

In [30]:
order_complexity = (
    df.groupby('Order Id')
      .agg(
          unique_products=('Product Card Id','nunique'),
          total_quantity=('Order Item Quantity','sum'),
          total_sales=('Sales','sum')
      )
      .reset_index()
)

df = df.merge(
    order_complexity,
    on='Order Id',
    how='left'
)

df['complex_order'] = (
    df['unique_products'] >= 3
).astype(int)

# RL State Features

In [31]:
state_features = [
    'Product Card Id',
    'Order Region',
    'Market',
    'Shipping Mode',
    'stocking_score',
    'regions_served',
    'likely_multi_region_stocked',
    'unique_products',
    'total_quantity',
    'shipping_delay',
    'margin_pct',
    'profit_per_item',
    'order_month',
    'order_weekday'
]

# Reward Features

In [32]:
df['reward'] = (
    df['Order Profit Per Order']
    + 100 * df['on_time_delivery']
    - 20 * df['late_delivery']
)

# Save Engineered Dataset

In [33]:
df.to_csv(
    "dataco_rl_feature_engineered.csv",
    index=False
)

print(df.shape)

(180519, 65)


# **STEP 3: ENCODING CATEGORICAL VARIABLES**

In [34]:

df = pd.read_csv('dataco_rl_feature_engineered.csv', encoding='latin1')


print("Dataset Loaded")
print(df.shape)

Dataset Loaded
(180519, 65)


In [35]:
one_hot_cols = [
    'Shipping Mode',
    'Market',
    'Order Region'
]

df = pd.get_dummies(
    df,
    columns=one_hot_cols,
    drop_first=False
)

df.to_csv(
    "dataco_rl_onehot.csv",
    index=False
)

print(df.shape)

(180519, 94)


# STEP 4: Train/test temporal split

In [48]:
import pandas as pd

df = pd.read_csv('dataco_rl_onehot.csv', encoding='latin1')

df['order date (DateOrders)'] = pd.to_datetime(
    df['order date (DateOrders)']
)

df = df.sort_values(
    'order date (DateOrders)'
)

n = len(df)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df.iloc[:train_end]
val_df = df.iloc[train_end:val_end]
test_df = df.iloc[val_end:]

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(126363, 94)
(27078, 94)
(27078, 94)


In [41]:
# Save datasets

train_df.to_csv(
    "dataco_rl_train.csv",
    index=False
)

val_df.to_csv(
    "dataco_rl_validation.csv",
    index=False
)

test_df.to_csv(
    "dataco_rl_test.csv",
    index=False
)

print("Files saved successfully")

Files saved successfully


# **STEP 4: NORMALIZATION ONLY FOR TRAINING DATA!!!!**

In [42]:
train_df = pd.read_csv("dataco_rl_train.csv")
val_df = pd.read_csv("dataco_rl_validation.csv")
test_df = pd.read_csv("dataco_rl_test.csv")


In [43]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

train_df[numeric_cols] = scaler.fit_transform(
    train_df[numeric_cols]
)

val_df[numeric_cols] = scaler.transform(
    val_df[numeric_cols]
)

test_df[numeric_cols] = scaler.transform(
    test_df[numeric_cols]
)

In [46]:
train_df.to_csv(
    "dataco_rl_train_scaled.csv",
    index=False
)

val_df.to_csv(
    "dataco_rl_validation_scaled.csv",
    index=False
)

test_df.to_csv(
    "dataco_rl_test_scaled.csv",
    index=False
)

print("Scaling complete.")
print("Features scaled:", len(numeric_cols))

Scaling complete.
Features scaled: 24


# STEP 5: STATE-SPACE DISCRETIZATION

# Feature Selection

In [44]:
continuous_features = [
    'stocking_score',
    'margin_pct',
    'profit_per_item',
    'shipping_delay',
    'actual_ship_days',
    'Sales',
    'Order Profit Per Order',
    'unique_products'
]

# Fit Bins Using Training Data Only

In [50]:
# Load scaled training data
train_df = pd.read_csv("dataco_rl_train_scaled.csv")

continuous_features = [
    'stocking_score',
    'margin_pct',
    'profit_per_item',
    'shipping_delay',
    'actual_ship_days',
    'Sales',
    'Order Profit Per Order',
    'unique_products'
]

bin_edges = {}

# Create 10 bins for each feature
for col in continuous_features:

    _, bins = pd.qcut(
        train_df[col],
        q=10,
        labels=False,
        retbins=True,
        duplicates='drop'
    )

    bin_edges[col] = bins

# Save bin definitions
joblib.dump(
    bin_edges,
    "state_bins.pkl"
)

print("Bins created successfully")

Bins created successfully


# Apply Bins to Train/Test Data

In [51]:
train_df = pd.read_csv("dataco_rl_train_scaled.csv")
test_df = pd.read_csv("dataco_rl_test_scaled.csv")

bin_edges = joblib.load("state_bins.pkl")

continuous_features = list(bin_edges.keys())

for col in continuous_features:

    train_df[col + "_bin"] = np.digitize(
        train_df[col],
        bin_edges[col][1:-1]
    )

    test_df[col + "_bin"] = np.digitize(
        test_df[col],
        bin_edges[col][1:-1]
    )

train_df.to_csv(
    "dataco_rl_train_discrete.csv",
    index=False
)

test_df.to_csv(
    "dataco_rl_test_discrete.csv",
    index=False
)

print("Discretization complete")

Discretization complete


# Build RL States

In [58]:
# Redefine state_columns with existing features in train_df
state_columns = [
    'Product Card Id',
    'Category Name',
    'shipping_mode_speed',
    'stocking_score_bin',
    'margin_pct_bin',
    'shipping_delay_bin',
    'profit_per_item_bin',
    'unique_products_bin'
]

train_df['state'] = train_df[state_columns] \
    .astype(str) \
    .agg('|'.join, axis=1)

# Create State IDs

In [59]:
state_encoder = LabelEncoder()

train_df['state_id'] = state_encoder.fit_transform(
    train_df['state']
)

joblib.dump(
    state_encoder,
    "state_encoder.pkl"
)

print(
    "Unique states:",
    train_df['state_id'].nunique()
)

Unique states: 14406


# Create Action IDs

In [64]:
candidate_regions = pd.read_csv(
    "candidate_fulfillment_regions.csv"
)

shipping_modes = [
    "Standard Class",
    "Second Class",
    "First Class",
    "Same Day"
]

actions = []

for region in candidate_regions["Order Region"].unique():
    for mode in shipping_modes:
        actions.append(f"{region}|{mode}")

from sklearn.preprocessing import LabelEncoder

action_encoder = LabelEncoder()

action_ids = action_encoder.fit_transform(actions)

action_mapping = pd.DataFrame({
    "action": actions,
    "action_id": action_ids
})

action_mapping.to_csv(
    "action_mapping.csv",
    index=False
)

In [67]:
action = (
    candidate_regions,
    shipping_modes
)

In [69]:
action_encoder = LabelEncoder()

# Reconstruct 'Shipping Mode' string from 'shipping_mode_speed'
shipping_mode_map_reverse = {
    1: 'Standard Class',
    2: 'Second Class',
    3: 'First Class',
    4: 'Same Day'
}
train_df['Shipping Mode_str'] = train_df['shipping_mode_speed'].map(shipping_mode_map_reverse)

# Reconstruct 'Order Region' string from one-hot encoded columns
order_region_cols = [col for col in train_df.columns if col.startswith('Order Region_')]
# Use idxmax to find the column with value 1, then strip the prefix
train_df['Order Region_str'] = train_df[order_region_cols].idxmax(axis=1).str.replace('Order Region_', '')

# Create the 'action' column by combining the reconstructed strings
train_df['action'] = train_df['Order Region_str'] + '|' + train_df['Shipping Mode_str']

# Now fit and transform the action column
train_df['action_id'] = action_encoder.fit_transform(
    train_df['action']
)

joblib.dump(
    action_encoder,
    "action_encoder.pkl"
)

print(
    "Unique actions:",
    train_df['action_id'].nunique()
)

Unique actions: 92


# Check State Space Size

In [70]:
print(
    "Unique states:",
    train_df['state'].nunique()
)

print(
    "Unique actions:",
    train_df['action'].nunique()
)

print(
    "Q-table size:",
    train_df['state'].nunique()
    *
    train_df['action'].nunique()
)

Unique states: 14406
Unique actions: 92
Q-table size: 1325352


# **STEP 6: Q-LEARNING IMPLEMENTATION**


# **STEP 7: SARSA IMPLEMENTATION**


# **STEP 8: COMPARISON OF PROFIT, REWARD, AND ON-TIME DELIVERY PERFORMANCE**


* State = information known BEFORE shipment decision
* Action = shipping method / route / priority decision
* Reward = profit − lateness penalty
* Next state = next order environment

Reward Function

Rt=α(Profit)−β(Late Shipment Penalty)

Where:

α controls profit importance

β controls delivery performance importance

A more detailed version:

Rt=αPt−βLt−γCt

Where:
Pt = profit

Lt = lateness indicator or delay days

Ct = shipping cost

# STEP 8: Q-learning or SARSA environment setup